### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import AutoTokenizer

from data.crisismmd import (
    load_crisismmd_annotations,
    build_fusion_dataframe,
    MultimodalCrisisDataset,
    make_eval_transforms,
)
from models.fusion_model import MultimodalFusionNetwork

### Load CrisisMMD and fusion dataframe

In [ ]:
root = "../data/CrisisMMD_v2.0"

combined = load_crisismmd_annotations(root)
fusion_df = build_fusion_dataframe(combined)

print(fusion_df["label"].value_counts())
train_df, test_df = train_test_split(fusion_df, test_size=0.2, random_state=42)

### Datasets and loaders

In [ ]:
text_model_dir = "../checkpoints/text_branch"          # from textbranch notebook
vision_weights = "../checkpoints/vision_brain.pth"     # from visionbranch notebook

tokenizer = AutoTokenizer.from_pretrained(text_model_dir)
img_tf = make_eval_transforms()  # no heavy augmentation for frozen encoders

train_dataset = MultimodalCrisisDataset(
    train_df, root_dir=root, tokenizer=tokenizer, image_transform=img_tf
)
test_dataset = MultimodalCrisisDataset(
    test_df, root_dir=root, tokenizer=tokenizer, image_transform=img_tf
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

### Build fusion model and criterion

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = MultimodalFusionNetwork(
    text_model_dir=text_model_dir,
    vision_weights_path=vision_weights,
    device=device,
).to(device)

# Class weights for imbalance: compute from fusion_df
count_safe_0 = (fusion_df["label"] == 0).sum()
count_disaster_1 = (fusion_df["label"] == 1).sum()
total = count_safe_0 + count_disaster_1

w0 = total / count_safe_0
w1 = total / count_disaster_1
class_weights = torch.tensor([w0, w1], dtype=torch.float32).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.fusion_classifier.parameters(), lr=1e-3)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.5)

EPOCHS = 10
train_loss_history = []
train_acc_history = []

### Training loop

In [ ]:
print("Starting Late Fusion Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    loop = tqdm(train_loader, leave=True)
    for batch in loop:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, images=images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        _, preds = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()

        acc = 100.0 * correct / total
        loop.set_description(f"Epoch [{epoch+1}/{EPOCHS}]")
        loop.set_postfix(loss=loss.item(), acc=acc)

    epoch_loss = running_loss / total
    epoch_acc = correct / total

    train_loss_history.append(epoch_loss)
    train_acc_history.append(epoch_acc)

    scheduler.step()

print("\nMultimodal Fusion Training complete.")


### Save fusion weights

In [ ]:
os.makedirs("../checkpoints", exist_ok=True)
fusion_path = "../checkpoints/fusion_brain.pth"
torch.save(model.state_dict(), fusion_path)
print(f"Saved fusion model to {fusion_path}")

### Plot training curves

In [ ]:
epochs = np.arange(1, EPOCHS + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(epochs, train_loss_history, marker="o")
ax[0].set_title("Fusion Training Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")

ax[1].plot(epochs, train_acc_history, marker="o")
ax[1].set_title("Fusion Training Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("Accuracy")

plt.tight_layout()
plt.show()

### Evaluation and confusion matrix

In [ ]:
model.eval()
all_preds = []
all_true  = []

print("Starting evaluation on unseen fusion test set...")

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        images = batch["image"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, images=images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

acc = accuracy_score(all_true, all_preds)
f1  = f1_score(all_true, all_preds, average="binary")

print("\n--- ULTIMATE MULTIMODAL FUSION RESULTS ---")
print(f"Test Accuracy: {acc * 100:.2f}%")
print(f"Test F1 Score: {f1:.4f}")

cm = confusion_matrix(all_true, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["Safe (0)", "Disaster (1)"])

fig, ax = plt.subplots(figsize=(6, 6))
disp.plot(cmap="Purples", ax=ax, colorbar=False)
plt.title("Late Fusion Model - Confusion Matrix (Test)", fontsize=14)
plt.grid(False)
plt.show()